In [ ]:
"""
negotiate_api.py  —  Subspace Negotiate API clearing-price optimizer
Uses XGBoost regressor to predict optimal clearing price given demand
signal (group size, platform, tenure). The Negotiate API sends batch
demand to content providers; this script finds the price point that
maximizes (GMV * acceptance_prob) — i.e., expected revenue.
"""
import numpy as np
import xgboost as xgb
from scipy.optimize import minimize_scalar
from dataclasses import dataclass
from typing import Optional

@dataclass
class DemandSignal:
    platform: str               # e.g. "netflix", "jiohotstar"
    group_size: int             # number of users in group
    avg_tenure_days: float     # avg days since first subscription
    churn_risk_score: float    # 0–1, from XGBoost churn model
    market_price_inr: float    # current provider list price
    platform_tier: str         # "premium"|"standard"|"basic"

class NegotiateAPIOptimizer:
    """
    Finds clearing price P* that maximizes:
      E[Revenue] = P * P(provider accepts P) * group_size
    Provider acceptance is modeled as logistic function of:
      - discount_depth (how far below market price)
      - committed_volume (group_size × tenure signal)
      - platform (some platforms have lower floor prices)
    """
    def __init__(self, acceptance_model_path: Optional[str] = None):
        # In prod: load pre-trained XGBoost model from S3/MLflow
        self.model = xgb.XGBRegressor() if not acceptance_model_path else \
                    xgb.XGBRegressor().load_model(acceptance_model_path)
        self.platform_floors = {
            "netflix": 0.60,       # provider never goes below 60% of list price
            "jiohotstar": 0.50,
            "spotify": 0.65,
            "youtube_premium": 0.70,
            "default": 0.55
        }

    def _acceptance_probability(self, offer_price: float, signal: DemandSignal) -> float:
        """Logistic model: P(accept) as function of discount depth + volume"""
        floor = self.platform_floors.get(signal.platform, self.platform_floors["default"])
        floor_price = signal.market_price_inr * floor
        if offer_price < floor_price:
            return 0.0
        # volume_factor: larger committed groups get better rates
        volume_factor = np.log1p(signal.group_size * signal.avg_tenure_days / 30)
        discount_ratio = (signal.market_price_inr - offer_price) / signal.market_price_inr
        # Logistic: steep acceptance cliff near floor, generous near market price
        z = 8 * ((offer_price - floor_price) / (signal.market_price_inr - floor_price)) \
            - 4 + 0.3 * volume_factor
        return float(1 / (1 + np.exp(-z)))

    def find_clearing_price(self, signal: DemandSignal) -> dict:
        """Returns optimal bid price and expected revenue"""
        def neg_expected_revenue(price):
            p_accept = self._acceptance_probability(price, signal)
            gmv = price * signal.group_size
            return -(gmv * p_accept)  # minimize negative = maximize revenue

        floor = signal.market_price_inr * \
                 self.platform_floors.get(signal.platform, 0.55)
        result = minimize_scalar(
            neg_expected_revenue,
            bounds=(floor, signal.market_price_inr),
            method="bounded"
        )
        p_star = result.x
        return {
            "clearing_price_inr": round(p_star, 2),
            "discount_pct": round((1 - p_star / signal.market_price_inr) * 100, 1),
            "expected_revenue_inr": round(-result.fun, 2),
            "acceptance_probability": round(
                self._acceptance_probability(p_star, signal), 3
            ),
        }

# ── Example call ─────────────────────────────────────────────
if __name__ == "__main__":
    optimizer = NegotiateAPIOptimizer()
    signal = DemandSignal(
        platform="jiohotstar", group_size=43,
        avg_tenure_days=87, churn_risk_score=0.31,
        market_price_inr=299, platform_tier="premium"
    )
    result = optimizer.find_clearing_price(signal)
    # → {'clearing_price_inr': 187.40, 'discount_pct': 37.3,
    #    'expected_revenue_inr': 7354.2, 'acceptance_probability': 0.812}
    print(result)